In [ ]:
# Cell 1: Import libraries and automatically locate both Kaggle datasets

from pathlib import Path
import os
import sys
import json
import pickle
import shutil
import numpy as np
import pandas as pd

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Available datasets inside /kaggle/input:\n")

for path in KAGGLE_INPUT.iterdir():
    print("-", path)

In [ ]:
# Cell 2: Automatically detect FACED and MSGM root folders

def find_folder(folder_name: str):
    matches = [
        path for path in KAGGLE_INPUT.rglob(folder_name)
        if path.is_dir()
    ]

    if not matches:
        raise FileNotFoundError(
            f"Could not find folder named: {folder_name}"
        )

    if len(matches) > 1:
        print(f"Multiple matches found for {folder_name}:")
        for match in matches:
            print("-", match)

    return matches[0]


FACED_ROOT = find_folder("FACED_EEG_GNN")
MSGM_ROOT = find_folder("MSGM-main")

print("FACED root:", FACED_ROOT)
print("MSGM root:", MSGM_ROOT)

print("\nFACED folders:")
for path in sorted(FACED_ROOT.iterdir()):
    print("-", path.name)

print("\nMSGM folders/files:")
for path in sorted(MSGM_ROOT.iterdir()):
    print("-", path.name)

In [ ]:
# Cell 3: Verify important FACED and MSGM files/folders

required_paths = {
    "FACED processed_data": FACED_ROOT / "processed_data",
    "FACED metadata": FACED_ROOT / "metadata",
    "FACED code": FACED_ROOT / "code",
    "FACED eeg_features": FACED_ROOT / "eeg_features",
    "MSGM model.py": MSGM_ROOT / "msgm" / "model.py",
    "MSGM preprocessing.py": MSGM_ROOT / "msgm" / "preprocessing.py",
    "MSGM config": MSGM_ROOT / "configs" / "paper_defaults.yaml",
    "MSGM requirements": MSGM_ROOT / "requirements.txt",
}

all_available = True

for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name:<30}: {exists} | {path}")

    if not exists:
        all_available = False

print("\nAll required paths available:", all_available)

In [ ]:
# Cell 4: Count and inspect FACED processed subject files

PROCESSED_DIR = FACED_ROOT / "processed_data"

subject_files = sorted(PROCESSED_DIR.glob("sub*.pkl"))

print("Number of processed subject files:", len(subject_files))

if not subject_files:
    raise FileNotFoundError(
        f"No subject files found inside: {PROCESSED_DIR}"
    )

print("\nFirst 10 files:")
for path in subject_files[:10]:
    print("-", path.name)

print("\nLast 10 files:")
for path in subject_files[-10:]:
    print("-", path.name)

expected_subjects = 123

if len(subject_files) == expected_subjects:
    print("\nSubject count is correct: 123")
else:
    print(
        f"\nWarning: Expected 123 subjects, "
        f"but found {len(subject_files)}"
    )

In [ ]:
# Cell 5: Load and deeply inspect one processed subject PKL file

sample_subject_path = subject_files[0]

with open(sample_subject_path, "rb") as file:
    sample_subject = pickle.load(file)

print("Sample file:", sample_subject_path)
print("Top-level Python type:", type(sample_subject))


def inspect_object(
    obj,
    name="root",
    depth=0,
    max_depth=4,
    max_items=10
):
    indent = "    " * depth

    if depth > max_depth:
        print(f"{indent}{name}: maximum depth reached")
        return

    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(
            f"{indent}{name}: dict, "
            f"number_of_keys={len(keys)}, "
            f"keys={keys[:max_items]}"
        )

        for key in keys[:max_items]:
            inspect_object(
                obj[key],
                name=str(key),
                depth=depth + 1,
                max_depth=max_depth,
                max_items=max_items
            )

    elif isinstance(obj, np.ndarray):
        print(
            f"{indent}{name}: ndarray, "
            f"shape={obj.shape}, "
            f"dtype={obj.dtype}, "
            f"min={np.nanmin(obj):.6f}, "
            f"max={np.nanmax(obj):.6f}"
        )

    elif isinstance(obj, pd.DataFrame):
        print(
            f"{indent}{name}: DataFrame, "
            f"shape={obj.shape}, "
            f"columns={obj.columns.tolist()}"
        )

    elif isinstance(obj, pd.Series):
        print(
            f"{indent}{name}: Series, "
            f"shape={obj.shape}, "
            f"name={obj.name}"
        )

    elif isinstance(obj, (list, tuple)):
        print(
            f"{indent}{name}: {type(obj).__name__}, "
            f"length={len(obj)}"
        )

        for index, value in enumerate(obj[:max_items]):
            inspect_object(
                value,
                name=f"[{index}]",
                depth=depth + 1,
                max_depth=max_depth,
                max_items=max_items
            )

    else:
        value_text = repr(obj)

        if len(value_text) > 200:
            value_text = value_text[:200] + "..."

        print(
            f"{indent}{name}: "
            f"{type(obj).__name__} = {value_text}"
        )


print("\nDetailed structure:\n")
inspect_object(sample_subject)

In [ ]:
# Cell 6: Inspect all FACED metadata files

METADATA_DIR = FACED_ROOT / "metadata"

metadata_filenames = [
    "Recording_info.csv",
    "Electrode_Location.xlsx",
    "Stimuli_info.xlsx",
    "Task_event.xlsx",
]

metadata_tables = {}

for filename in metadata_filenames:
    path = METADATA_DIR / filename

    print("\n" + "=" * 90)
    print("File:", filename)
    print("Path:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    try:
        if path.suffix.lower() == ".csv":
            dataframe = pd.read_csv(path)
        else:
            dataframe = pd.read_excel(path)

        metadata_tables[filename] = dataframe

        print("Shape:", dataframe.shape)
        print("Columns:", dataframe.columns.tolist())
        print("\nFirst five rows:")
        display(dataframe.head())

    except Exception as error:
        print("Could not read file.")
        print("Error:", error)

In [ ]:
# Cell 7: Inspect all sheets inside Excel metadata files

excel_files = [
    METADATA_DIR / "Electrode_Location.xlsx",
    METADATA_DIR / "Stimuli_info.xlsx",
    METADATA_DIR / "Task_event.xlsx",
]

for path in excel_files:
    if not path.exists():
        continue

    print("\n" + "=" * 90)
    print("Workbook:", path.name)

    workbook = pd.ExcelFile(path)

    print("Sheet names:", workbook.sheet_names)

    for sheet_name in workbook.sheet_names:
        dataframe = pd.read_excel(path, sheet_name=sheet_name)

        print(f"\nSheet: {sheet_name}")
        print("Shape:", dataframe.shape)
        print("Columns:", dataframe.columns.tolist())

        display(dataframe.head())

In [ ]:
# Cell 8: Inspect precomputed DE and PSD feature folders

EEG_FEATURES_DIR = FACED_ROOT / "eeg_features"

print("Feature folders:")

for path in sorted(EEG_FEATURES_DIR.iterdir()):
    print("-", path.name)

for feature_name in ["DE", "PSD"]:
    feature_dir = EEG_FEATURES_DIR / feature_name

    print("\n" + "=" * 90)
    print("Feature type:", feature_name)
    print("Path exists:", feature_dir.exists())

    if not feature_dir.exists():
        continue

    feature_files = sorted(feature_dir.glob("sub*.pkl"))

    print("Number of subject files:", len(feature_files))

    if feature_files:
        print("Sample file:", feature_files[0].name)

        with open(feature_files[0], "rb") as file:
            sample_feature = pickle.load(file)

        inspect_object(
            sample_feature,
            name=f"{feature_name}_sample",
            max_depth=3
        )

In [ ]:
# Cell 9: List original FACED Python code files

FACED_CODE_DIR = FACED_ROOT / "code"

python_files = sorted(FACED_CODE_DIR.rglob("*.py"))

print("Total Python files:", len(python_files))
print()

for path in python_files:
    print(path.relative_to(FACED_ROOT))

In [ ]:
# Cell 10: Find high-priority FACED scripts

priority_names = {
    "load_data.py",
    "main_classify.py",
    "main_pretrain.py",
    "train_utils.py",
    "running_norm.py",
    "features_extract.py",
    "reorder_vids.py",
    "io_utils.py",
    "model.py",
    "run_code.py",
}

priority_files = [
    path for path in python_files
    if path.name in priority_names
]

print("Priority scripts found:\n")

for path in priority_files:
    print("-", path.relative_to(FACED_ROOT))

In [ ]:
# Cell 11: Search FACED code for labels, folds, normalization and metrics

search_terms = [
    "valence",
    "label",
    "cls2",
    "binary",
    "neutral",
    "fold",
    "leave",
    "subject",
    "f1",
    "accuracy",
    "normalize",
    "running_norm",
]

matches = []

for python_file in python_files:
    try:
        text = python_file.read_text(
            encoding="utf-8",
            errors="ignore"
        )

        lower_text = text.lower()

        found_terms = [
            term for term in search_terms
            if term.lower() in lower_text
        ]

        if found_terms:
            matches.append(
                {
                    "file": str(
                        python_file.relative_to(FACED_ROOT)
                    ),
                    "matched_terms": ", ".join(found_terms),
                }
            )

    except Exception as error:
        print(f"Could not read {python_file}: {error}")

matches_df = pd.DataFrame(matches)

print("Files containing important keywords:")
display(matches_df)

In [ ]:
# Cell 12: Copy official MSGM repository into writable Kaggle working folder

LOCAL_MSGM_ROOT = KAGGLE_WORKING / "MSGM-main"

if LOCAL_MSGM_ROOT.exists():
    shutil.rmtree(LOCAL_MSGM_ROOT)

shutil.copytree(
    MSGM_ROOT,
    LOCAL_MSGM_ROOT
)

print("Official MSGM code copied to:")
print(LOCAL_MSGM_ROOT)

print("\nCopied files:")

for path in sorted(LOCAL_MSGM_ROOT.rglob("*")):
    if path.is_file():
        print("-", path.relative_to(LOCAL_MSGM_ROOT))

In [ ]:
# Cell 13: Install required packages and add MSGM to Python path

!pip install -q pyyaml scipy openpyxl

if str(LOCAL_MSGM_ROOT) not in sys.path:
    sys.path.insert(0, str(LOCAL_MSGM_ROOT))

print("MSGM path added to sys.path:")
print(sys.path[0])

In [ ]:
# Cell 14: Test official MSGM package import

import importlib

try:
    import msgm
    importlib.reload(msgm)

    print("MSGM package imported successfully.")
    print("MSGM package location:", msgm.__file__)

except Exception as error:
    print("MSGM import failed.")
    print("Error type:", type(error).__name__)
    print("Error message:", error)

In [ ]:
# Cell 15: Inspect available classes and functions in MSGM modules

import inspect
import msgm.model as msgm_model
import msgm.preprocessing as msgm_preprocessing

print("Classes/functions inside msgm.model:\n")

for name, obj in inspect.getmembers(msgm_model):
    if (
        inspect.isclass(obj)
        or inspect.isfunction(obj)
    ):
        if obj.__module__ == msgm_model.__name__:
            print("-", name)

print("\nClasses/functions inside msgm.preprocessing:\n")

for name, obj in inspect.getmembers(msgm_preprocessing):
    if (
        inspect.isclass(obj)
        or inspect.isfunction(obj)
    ):
        if obj.__module__ == msgm_preprocessing.__name__:
            print("-", name)

In [ ]:
# Cell 16: Run the official MSGM smoke-forward example

smoke_script = LOCAL_MSGM_ROOT / "examples" / "smoke_forward.py"

print("Smoke test file:", smoke_script)
print("Exists:", smoke_script.exists())

if smoke_script.exists():
    !python "{smoke_script}"
else:
    print("smoke_forward.py was not found.")

In [ ]:
# Cell 17: Confirm the processed-data schema across all 123 subjects

import pickle
import numpy as np
import pandas as pd
from pathlib import Path

EXPECTED_SHAPE = (28, 32, 7500)
schema_rows = []

for path in subject_files:
    with open(path, "rb") as file:
        array = pickle.load(file)

    schema_rows.append({
        "subject": path.stem,
        "shape": tuple(array.shape) if isinstance(array, np.ndarray) else None,
        "dtype": str(array.dtype) if isinstance(array, np.ndarray) else type(array).__name__,
        "finite": bool(np.isfinite(array).all()) if isinstance(array, np.ndarray) else False,
        "matches_expected_shape": isinstance(array, np.ndarray) and tuple(array.shape) == EXPECTED_SHAPE,
    })

schema_df = pd.DataFrame(schema_rows)

print("Subjects inspected:", len(schema_df))
print("All arrays have expected shape:", schema_df["matches_expected_shape"].all())
print("All arrays contain finite values:", schema_df["finite"].all())
print("\nShape counts:")
display(schema_df["shape"].value_counts().rename_axis("shape").reset_index(name="count"))

if not schema_df["matches_expected_shape"].all():
    display(schema_df.loc[~schema_df["matches_expected_shape"]])
    raise ValueError("At least one processed subject does not have shape (28, 32, 7500).")

In [ ]:
# Cell 18: Build the 28-trial binary label table from Stimuli_info.xlsx

stimuli_path = METADATA_DIR / "Stimuli_info.xlsx"
stimuli_raw = pd.read_excel(stimuli_path)

# Keep only rows whose Video index can be interpreted as an integer from 1 to 28.
stimuli = stimuli_raw.copy()
stimuli["video_index_numeric"] = pd.to_numeric(stimuli["Video index"], errors="coerce")
stimuli = stimuli.loc[stimuli["video_index_numeric"].between(1, 28)].copy()
stimuli["video_index"] = stimuli["video_index_numeric"].astype(int)
stimuli = stimuli.sort_values("video_index").drop_duplicates("video_index")

if len(stimuli) != 28:
    raise ValueError(f"Expected 28 valid stimulus rows, found {len(stimuli)}.")

stimuli["valence_text"] = (
    stimuli["Valence"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Reconstructed binary rule:
# Negative -> 0, Positive -> 1, Neutral -> excluded.
label_map = {"negative": 0, "positive": 1}
stimuli["binary_label"] = stimuli["valence_text"].map(label_map)
stimuli["include_binary"] = stimuli["binary_label"].notna()

label_table = stimuli[
    [
        "video_index",
        "Duration（s）",
        "Valence",
        "Targeted Emotion",
        "binary_label",
        "include_binary",
    ]
].reset_index(drop=True)

print("All 28 stimuli:")
display(label_table)

print("\nBinary trial counts after excluding neutral:")
display(
    label_table.loc[label_table["include_binary"], "binary_label"]
    .value_counts()
    .sort_index()
    .rename(index={0: "negative", 1: "positive"})
)

print("\nExcluded trials:")
display(label_table.loc[~label_table["include_binary"]])

BINARY_TRIAL_INDICES = (
    label_table.loc[label_table["include_binary"], "video_index"].astype(int).to_numpy() - 1
)
BINARY_TRIAL_LABELS = (
    label_table.loc[label_table["include_binary"], "binary_label"].astype(int).to_numpy()
)

print("Included binary trials:", len(BINARY_TRIAL_INDICES))
print("Trial indices (0-based):", BINARY_TRIAL_INDICES.tolist())

In [ ]:
# Cell 19: Define reproduction configuration without hard-coding scale lengths

from dataclasses import dataclass, asdict
from pathlib import Path
import json


@dataclass
class ReproductionConfig:
    # Dataset
    sampling_rate: int = 250
    num_channels: int = 32
    num_features: int = 7
    num_classes: int = 2

    # First-level segmentation
    first_window_seconds: float = 20.0
    first_hop_seconds: float = 4.0

    # Second-level segmentation
    sub_window_seconds: tuple = (
        4.0,
        3.0,
        2.0,
        1.0,
    )
    sub_overlap: float = 0.75

    # Do not hard-code here.
    # Cell 20 will detect the exact sequence lengths
    # produced by the official preprocessing code.
    scale_lengths: tuple = ()

    # Architecture
    hidden_dim: int = 32
    graph_layers: tuple = (1, 2)
    chebyshev_order: int = 4
    mamba_layers: int = 1
    mamba_state_dim: int = 16
    mamba_conv_kernel: int = 4
    mamba_expand: int = 2
    dropout: float = 0.25

    # Training
    learning_rate: float = 3e-4
    weight_decay: float = 1e-2
    label_smoothing: float = 0.1
    batch_size: int = 32
    epochs: int = 30
    early_stopping_patience: int = 5
    random_seed: int = 42

    # Reconstructed protocol
    num_test_folds: int = 10
    fold_mode: str = "paper_close_10x12"
    validation_mode: str = "subject_wise"
    neutral_policy: str = "exclude"


CONFIG = ReproductionConfig()

WORK_DIR = Path("/kaggle/working/faced_msgm")
FEATURE_DIR = WORK_DIR / "rpsd_subjects"
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
RESULT_DIR = WORK_DIR / "results"

for directory in [
    WORK_DIR,
    FEATURE_DIR,
    CHECKPOINT_DIR,
    RESULT_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Base configuration created.")
print("Scale lengths will be detected automatically in Cell 20.")

In [ ]:
# Cell 20: Detect and validate actual official MSGM rPSD sequence lengths

import pickle
import numpy as np
import json
from dataclasses import asdict

from msgm.preprocessing import make_multiscale_rpsd


# Load one subject
with open(subject_files[0], "rb") as file:
    subject0 = pickle.load(file)

print("Subject file:", subject_files[0].name)
print("Subject array shape:", subject0.shape)


# First trial: (channels, time)
trial0_channel_time = subject0[0]

if trial0_channel_time.shape[0] != CONFIG.num_channels:
    raise ValueError(
        f"Expected {CONFIG.num_channels} channels, "
        f"found shape {trial0_channel_time.shape}"
    )


# Convert to (time, channels)
trial0_time_channel = trial0_channel_time.T

print("Trial input shape:", trial0_time_channel.shape)


# Official preprocessing
pilot_scales = make_multiscale_rpsd(
    raw_eeg=trial0_time_channel,
    sampling_rate=CONFIG.sampling_rate,
    first_window_seconds=CONFIG.first_window_seconds,
    first_hop_seconds=CONFIG.first_hop_seconds,
    sub_window_seconds=CONFIG.sub_window_seconds,
    sub_overlap=CONFIG.sub_overlap,
)

if len(pilot_scales) != len(CONFIG.sub_window_seconds):
    raise ValueError(
        f"Expected {len(CONFIG.sub_window_seconds)} scales, "
        f"found {len(pilot_scales)}"
    )


# Detect actual sequence lengths dynamically
actual_scale_lengths = tuple(
    int(scale.shape[1])
    for scale in pilot_scales
)

CONFIG.scale_lengths = actual_scale_lengths

print("\nDetected scale lengths:", CONFIG.scale_lengths)


# Print all scale shapes
for window_seconds, scale in zip(
    CONFIG.sub_window_seconds,
    pilot_scales,
):
    print(
        f"{window_seconds:.0f}s scale -> "
        f"shape={tuple(scale.shape)}, "
        f"sequence length={scale.shape[1]}"
    )


# Validate dimensions and values
for scale_index, scale in enumerate(pilot_scales):
    if scale.ndim != 4:
        raise ValueError(
            f"Scale {scale_index} must be 4D, "
            f"found shape {scale.shape}"
        )

    if scale.shape[2] != CONFIG.num_channels:
        raise ValueError(
            f"Scale {scale_index} has incorrect channel count: "
            f"{scale.shape[2]}"
        )

    if scale.shape[3] != CONFIG.num_features:
        raise ValueError(
            f"Scale {scale_index} has incorrect feature count: "
            f"{scale.shape[3]}"
        )

    if not np.isfinite(scale).all():
        raise ValueError(
            f"Scale {scale_index} contains NaN or infinity."
        )


# Save corrected configuration after detecting scale lengths
config_path = WORK_DIR / "reproduction_config.json"

with open(
    config_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        asdict(CONFIG),
        file,
        indent=2,
    )


print("\nPilot rPSD validation passed successfully.")
print("Final scale lengths:", CONFIG.scale_lengths)
print("Configuration saved to:", config_path)

## GPU timing

**Do not enable GPU yet.** Cells 21–25 perform SciPy/NumPy preprocessing and primarily use the CPU.

Run Cells 21–25 with the CPU accelerator. After the rPSD cache is complete, save the notebook output or create a Kaggle dataset from `/kaggle/working/faced_msgm`. Then enable a GPU before Cell 29. Changing the accelerator may restart the session, so the safest workflow is:

1. Finish and save the rPSD cache.
2. Create a Kaggle Dataset from the cache, or save a notebook version with outputs.
3. Enable GPU.
4. Rerun setup/import cells and continue from the cached features.

In [ ]:
# Cell 21: Extract and cache official MSGM rPSD features subject by subject (CPU)

from tqdm.auto import tqdm
import gc

def extract_subject_rpsd(subject_path: Path, output_path: Path) -> None:
    with open(subject_path, "rb") as file:
        subject_array = pickle.load(file)

    if tuple(subject_array.shape) != EXPECTED_SHAPE:
        raise ValueError(f"{subject_path.name}: unexpected shape {subject_array.shape}")

    scale_buffers = [[] for _ in CONFIG.scale_lengths]
    sample_labels = []
    sample_trials = []

    for trial_index, trial_label in zip(BINARY_TRIAL_INDICES, BINARY_TRIAL_LABELS):
        raw_eeg = np.asarray(subject_array[trial_index].T, dtype=np.float64)

        scales = make_multiscale_rpsd(
            raw_eeg=raw_eeg,
            sampling_rate=CONFIG.sampling_rate,
            first_window_seconds=CONFIG.first_window_seconds,
            first_hop_seconds=CONFIG.first_hop_seconds,
            sub_window_seconds=CONFIG.sub_window_seconds,
            sub_overlap=CONFIG.sub_overlap,
        )

        first_segment_count = scales[0].shape[0]

        for buffer, scale in zip(scale_buffers, scales):
            buffer.append(scale.astype(np.float32, copy=False))

        sample_labels.extend([int(trial_label)] * first_segment_count)
        sample_trials.extend([int(trial_index)] * first_segment_count)

    arrays = {
        f"scale_{length}": np.concatenate(buffer, axis=0)
        for length, buffer in zip(CONFIG.scale_lengths, scale_buffers)
    }
    arrays["labels"] = np.asarray(sample_labels, dtype=np.int64)
    arrays["trial_indices"] = np.asarray(sample_trials, dtype=np.int16)
    arrays["subject_id"] = np.asarray(subject_path.stem)

    np.savez_compressed(output_path, **arrays)


failed_subjects = []

for subject_path in tqdm(subject_files, desc="Extracting subject rPSD"):
    output_path = FEATURE_DIR / f"{subject_path.stem}.npz"

    if output_path.exists():
        continue

    try:
        extract_subject_rpsd(subject_path, output_path)
    except Exception as error:
        failed_subjects.append((subject_path.name, repr(error)))

    gc.collect()

print("Cached subjects:", len(list(FEATURE_DIR.glob("sub*.npz"))))
print("Failed subjects:", len(failed_subjects))

if failed_subjects:
    display(pd.DataFrame(failed_subjects, columns=["subject", "error"]))
    raise RuntimeError("Some subjects failed during rPSD extraction.")

In [ ]:
# Cell 22: Validate every cached rPSD subject file

validation_rows = []

for path in sorted(FEATURE_DIR.glob("sub*.npz")):
    with np.load(path, allow_pickle=False) as data:
        labels = data["labels"]

        row = {
            "subject": path.stem,
            "samples": len(labels),
            "negative": int((labels == 0).sum()),
            "positive": int((labels == 1).sum()),
            "finite": True,
        }

        for length in CONFIG.scale_lengths:
            array = data[f"scale_{length}"]
            row[f"shape_{length}"] = tuple(array.shape)
            row["finite"] = row["finite"] and bool(np.isfinite(array).all())

        validation_rows.append(row)

cache_df = pd.DataFrame(validation_rows)

print("Cached subject files:", len(cache_df))
print("All finite:", cache_df["finite"].all())
print("\nSample count distribution:")
display(cache_df["samples"].value_counts().rename_axis("samples").reset_index(name="subjects"))
print("\nClass totals:")
print(cache_df[["negative", "positive"]].sum())

display(cache_df.head())

if len(cache_df) != 123:
    raise ValueError("Expected 123 cached subjects.")
if not cache_df["finite"].all():
    raise ValueError("Non-finite values found in cached rPSD data.")

In [ ]:
# Cell 23: Create the sample manifest

manifest_rows = []

for path in sorted(FEATURE_DIR.glob("sub*.npz")):
    with np.load(path, allow_pickle=False) as data:
        labels = data["labels"]
        trials = data["trial_indices"]

        for sample_index, (label, trial_index) in enumerate(zip(labels, trials)):
            manifest_rows.append({
                "subject_id": path.stem,
                "feature_file": str(path),
                "sample_index": sample_index,
                "trial_index_0based": int(trial_index),
                "video_index_1based": int(trial_index) + 1,
                "label": int(label),
            })

manifest_df = pd.DataFrame(manifest_rows)
manifest_path = WORK_DIR / "sample_manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print("Manifest shape:", manifest_df.shape)
print("Unique subjects:", manifest_df["subject_id"].nunique())
print("Class counts:")
display(manifest_df["label"].value_counts().sort_index())
display(manifest_df.head())
print("Saved:", manifest_path)

In [ ]:
# Cell 24: Create deterministic cross-subject folds

from sklearn.model_selection import train_test_split

all_subject_ids = sorted(manifest_df["subject_id"].unique().tolist())
rng = np.random.default_rng(CONFIG.random_seed)
shuffled_subjects = np.asarray(all_subject_ids)
rng.shuffle(shuffled_subjects)

# Paper-close reconstruction:
# 10 folds × 12 test subjects = 120 test subjects.
# The final 3 subjects are never test subjects and remain eligible for train/validation.
# This is transparent but NOT confirmed as the authors' exact fold assignment.
test_pool = shuffled_subjects[:120]
always_train_pool = shuffled_subjects[120:]
test_folds = [list(x) for x in np.split(test_pool, CONFIG.num_test_folds)]

folds = []

for fold_index, test_subjects in enumerate(test_folds, start=1):
    remaining = [s for s in all_subject_ids if s not in set(test_subjects)]

    if CONFIG.validation_mode == "subject_wise":
        train_subjects, val_subjects = train_test_split(
            remaining,
            test_size=0.10,
            random_state=CONFIG.random_seed + fold_index,
            shuffle=True,
        )
    else:
        # Segment-wise validation is implemented later if explicitly selected.
        train_subjects = remaining
        val_subjects = []

    folds.append({
        "fold": fold_index,
        "train_subjects": sorted(train_subjects),
        "val_subjects": sorted(val_subjects),
        "test_subjects": sorted(test_subjects),
    })

fold_path = WORK_DIR / "subject_folds.json"
with open(fold_path, "w", encoding="utf-8") as file:
    json.dump(
        {
            "description": (
                "Reconstructed paper-close folds; exact MSGM subject IDs were not published."
            ),
            "random_seed": CONFIG.random_seed,
            "always_train_pool": always_train_pool.tolist(),
            "folds": folds,
        },
        file,
        indent=2,
    )

fold_summary = pd.DataFrame([
    {
        "fold": fold["fold"],
        "train_subjects": len(fold["train_subjects"]),
        "val_subjects": len(fold["val_subjects"]),
        "test_subjects": len(fold["test_subjects"]),
    }
    for fold in folds
])

display(fold_summary)
print("Always-train subjects:", always_train_pool.tolist())
print("Saved:", fold_path)

In [ ]:
# Cell 25: Save a compact preprocessing summary before switching to GPU

preprocessing_summary = {
    "processed_subjects": len(subject_files),
    "cached_subjects": len(list(FEATURE_DIR.glob("sub*.npz"))),
    "processed_shape_per_subject": list(EXPECTED_SHAPE),
    "binary_trials_per_subject": int(len(BINARY_TRIAL_INDICES)),
    "neutral_policy": CONFIG.neutral_policy,
    "scale_lengths": list(CONFIG.scale_lengths),
    "manifest_samples": int(len(manifest_df)),
    "fold_file": str(fold_path),
    "feature_directory": str(FEATURE_DIR),
}

summary_path = WORK_DIR / "preprocessing_summary.json"
with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(preprocessing_summary, file, indent=2)

print(json.dumps(preprocessing_summary, indent=2))
print("\nCPU preprocessing stage is complete.")
print("Save this notebook version or package /kaggle/working/faced_msgm as a Kaggle Dataset.")

# GPU stage

Enable a Kaggle GPU **before Cell 29**.

Recommended accelerator: **GPU T4 x2 is not necessary; one T4/P100 is sufficient.**

After changing the accelerator, rerun the setup/import cells needed to restore paths and imports. Make sure the cached `rpsd_subjects` directory is still available. If it is not, attach the cache as a Kaggle Dataset and update `FEATURE_DIR`.

In [ ]:
# Cell 26: GPU compatibility check and safe device selection

import os
import random
import numpy as np
import torch


def set_global_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_global_seed(CONFIG.random_seed)


print("PyTorch version:", torch.__version__)
print("PyTorch CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())


if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)

    print("GPU name:", gpu_name)
    print("GPU compute capability:", gpu_capability)

    try:
        # Test whether the installed PyTorch build can execute
        # a basic CUDA operation on this GPU.
        test_tensor = torch.tensor(
            [1.0, 2.0, 3.0],
            device="cuda",
        )

        test_result = test_tensor.mean()

        torch.cuda.synchronize()

        print("CUDA compatibility test passed.")
        print("CUDA test result:", test_result.item())

        DEVICE = torch.device("cuda")

    except Exception as error:
        print("\nCUDA exists but is incompatible with this runtime.")
        print("Error:", error)

        DEVICE = torch.device("cpu")

        print(
            "\nUsing CPU fallback. "
            "For training, restart the notebook with a T4 GPU."
        )

else:
    DEVICE = torch.device("cpu")

    print(
        "No GPU detected. "
        "Enable a Kaggle T4 GPU before full training."
    )


print("\nSelected device:", DEVICE)

In [ ]:
# Cell 27: Define a lazy subject-level rPSD Dataset

from pathlib import Path
import numpy as np
import torch

from torch.utils.data import Dataset, DataLoader


class FACEDRPSDDataset(Dataset):
    def __init__(self, subject_ids, feature_dir: Path):
        self.feature_dir = Path(feature_dir)
        self.subject_ids = list(subject_ids)

        self.index = []

        self._cache_subject = None
        self._cache_data = None

        for subject_id in self.subject_ids:
            path = self.feature_dir / f"{subject_id}.npz"

            if not path.exists():
                raise FileNotFoundError(
                    f"Feature file not found: {path}"
                )

            with np.load(path, allow_pickle=False) as data:
                sample_count = len(data["labels"])

            self.index.extend(
                (subject_id, sample_index)
                for sample_index in range(sample_count)
            )

    def __len__(self):
        return len(self.index)

    def _load_subject(self, subject_id):
        if self._cache_subject != subject_id:
            path = self.feature_dir / f"{subject_id}.npz"

            with np.load(path, allow_pickle=False) as loaded:
                self._cache_data = {
                    f"scale_{length}": loaded[
                        f"scale_{length}"
                    ].copy()
                    for length in CONFIG.scale_lengths
                }

                self._cache_data["labels"] = (
                    loaded["labels"].copy()
                )

            self._cache_subject = subject_id

        return self._cache_data

    def __getitem__(self, index):
        subject_id, sample_index = self.index[index]

        data = self._load_subject(subject_id)

        scales = [
            torch.from_numpy(
                data[f"scale_{length}"][sample_index]
            ).float()
            for length in CONFIG.scale_lengths
        ]

        label = torch.tensor(
            int(data["labels"][sample_index]),
            dtype=torch.long,
        )

        return scales, label, subject_id


def create_fold_loaders(fold, batch_size=None):
    if batch_size is None:
        batch_size = CONFIG.batch_size

    train_dataset = FACEDRPSDDataset(
        fold["train_subjects"],
        FEATURE_DIR,
    )

    val_dataset = FACEDRPSDDataset(
        fold["val_subjects"],
        FEATURE_DIR,
    )

    test_dataset = FACEDRPSDDataset(
        fold["test_subjects"],
        FEATURE_DIR,
    )

    train_generator = torch.Generator()
    train_generator.manual_seed(
        CONFIG.random_seed + fold["fold"]
    )

    use_pin_memory = (
        "DEVICE" in globals()
        and DEVICE.type == "cuda"
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=True,
        num_workers=0,
        pin_memory=use_pin_memory,
        generator=train_generator,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=use_pin_memory,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        num_workers=0,
        pin_memory=use_pin_memory,
    )

    return train_loader, val_loader, test_loader


print("FACEDRPSDDataset and DataLoader functions defined successfully.")
print("Detected scale lengths:", CONFIG.scale_lengths)
print("Feature directory:", FEATURE_DIR)

In [ ]:
# Cell 28: Validate one DataLoader batch before model training

train_loader, val_loader, test_loader = create_fold_loaders(folds[0], CONFIG.batch_size)

batch_scales, batch_labels, batch_subjects = next(iter(train_loader))

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))
print("Labels shape:", batch_labels.shape)
print("First batch subjects:", list(batch_subjects[:5]))

for length, tensor in zip(CONFIG.scale_lengths, batch_scales):
    print(f"Scale {length}: {tuple(tensor.shape)}")

expected_shapes = [
    (CONFIG.batch_size, length, CONFIG.num_channels, CONFIG.num_features)
    for length in CONFIG.scale_lengths
]
actual_shapes = [tuple(tensor.shape) for tensor in batch_scales]

if actual_shapes != expected_shapes:
    raise ValueError(f"Unexpected batch shapes: {actual_shapes}")

In [ ]:
# Cell 29: Build and test the official 32-channel MSGM model

import gc
import torch

from msgm import MSGM, count_parameters


def build_model(device):
    model = MSGM(
        num_channels=CONFIG.num_channels,
        num_features=CONFIG.num_features,
        num_classes=CONFIG.num_classes,
        hidden_dim=CONFIG.hidden_dim,
        graph_layers=CONFIG.graph_layers,
        chebyshev_order=CONFIG.chebyshev_order,
        scale_lengths=CONFIG.scale_lengths,
        mamba_layers=CONFIG.mamba_layers,
        mamba_state_dim=CONFIG.mamba_state_dim,
        mamba_conv_kernel=CONFIG.mamba_conv_kernel,
        mamba_expand=CONFIG.mamba_expand,
        dropout=CONFIG.dropout,
    )

    model = model.to(device)

    return model


# Clear old GPU state
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


print("Selected device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Compute capability:",
        torch.cuda.get_device_capability(0),
    )


model = build_model(DEVICE)

print("\nModel class:", model.__class__.__name__)
print("Trainable parameters:", count_parameters(model))
print(
    "Model parameter device:",
    next(model.parameters()).device,
)


# Select two samples from each temporal scale
smoke_inputs = []

for scale_index, tensor in enumerate(batch_scales):
    smoke_tensor = tensor[:2].contiguous().to(
        DEVICE,
        non_blocking=False,
    )

    smoke_inputs.append(smoke_tensor)

    print(
        f"Input scale {scale_index + 1}: "
        f"shape={tuple(smoke_tensor.shape)}, "
        f"device={smoke_tensor.device}, "
        f"dtype={smoke_tensor.dtype}"
    )


model.eval()

try:
    with torch.no_grad():
        # Official MSGM forward expects each scale tensor
        # as a separate positional argument.
        smoke_logits = model(*smoke_inputs)

        if DEVICE.type == "cuda":
            torch.cuda.synchronize()

except RuntimeError as error:
    print("\nModel forward test failed.")
    print("Runtime error:", error)

    if "no kernel image" in str(error).lower():
        raise RuntimeError(
            "The selected Kaggle GPU is incompatible with the "
            "installed PyTorch/CUDA build. Restart the notebook "
            "with a Tesla T4 GPU."
        ) from error

    raise


print("\nGPU smoke logits shape:", tuple(smoke_logits.shape))
print("GPU smoke logits device:", smoke_logits.device)
print("GPU smoke logits:")
print(smoke_logits)


expected_output_shape = (2, CONFIG.num_classes)

if tuple(smoke_logits.shape) != expected_output_shape:
    raise ValueError(
        "Unexpected MSGM output shape. "
        f"Expected {expected_output_shape}, "
        f"found {tuple(smoke_logits.shape)}."
    )


print("\nMSGM GPU forward-pass validation succeeded.")

In [ ]:
# Cell 30: Define training, evaluation, and detailed runtime functions

import gc
import time
import numpy as np
import pandas as pd
import torch

from dataclasses import asdict
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
)


def synchronize_device():
    """Wait until all pending GPU operations finish before timing."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()


def format_seconds(seconds):
    """Return a readable runtime string."""
    seconds = float(seconds)

    if seconds < 60:
        return f"{seconds:.2f} sec"

    minutes = seconds / 60

    if minutes < 60:
        return f"{minutes:.2f} min"

    return f"{minutes / 60:.2f} hr"


def move_scales_to_device(scales):
    """Move all temporal-scale tensors to CPU/GPU."""
    return [
        scale.to(
            DEVICE,
            non_blocking=(DEVICE.type == "cuda"),
        )
        for scale in scales
    ]


def compute_metrics(y_true, y_pred):
    """Calculate accuracy and multiple F1 variants."""

    return {
        "accuracy": accuracy_score(y_true, y_pred),

        "f1_binary": f1_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),

        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),

        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),

        "precision_binary": precision_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),

        "recall_binary": recall_score(
            y_true,
            y_pred,
            average="binary",
            zero_division=0,
        ),
    }


def run_epoch(
    model,
    loader,
    criterion,
    optimizer=None,
):
    """
    Run one complete training, validation, or testing pass.

    Returns:
        metrics
        y_true
        y_pred
        elapsed_seconds
    """

    is_training = optimizer is not None

    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_samples = 0

    y_true = []
    y_pred = []

    synchronize_device()
    epoch_start = time.perf_counter()

    for scales, labels, _ in loader:
        scales = move_scales_to_device(scales)

        labels = labels.to(
            DEVICE,
            non_blocking=(DEVICE.type == "cuda"),
        )

        if is_training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_training):
            # Four temporal scales are passed separately.
            logits = model(*scales)

            loss = criterion(
                logits,
                labels,
            )

            if is_training:
                loss.backward()
                optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        predictions = logits.argmax(dim=1)

        y_true.extend(
            labels.detach().cpu().numpy().tolist()
        )

        y_pred.extend(
            predictions.detach().cpu().numpy().tolist()
        )

    synchronize_device()
    elapsed_seconds = time.perf_counter() - epoch_start

    if total_samples == 0:
        raise RuntimeError(
            "The DataLoader produced zero samples."
        )

    metrics = compute_metrics(
        y_true,
        y_pred,
    )

    metrics["loss"] = (
        total_loss / total_samples
    )

    metrics["n_samples"] = total_samples
    metrics["runtime_seconds"] = elapsed_seconds
    metrics["runtime_minutes"] = elapsed_seconds / 60

    return (
        metrics,
        np.asarray(y_true),
        np.asarray(y_pred),
        elapsed_seconds,
    )


def train_one_fold(
    fold,
    max_epochs=None,
):
    """
    Train, validate, and test MSGM for one cross-subject fold.

    Timings returned:
        model setup time
        each training epoch time
        each validation epoch time
        total training time
        total validation time
        checkpoint loading time
        test time
        complete fold runtime
    """

    fold_number = int(fold["fold"])

    set_global_seed(
        CONFIG.random_seed + fold_number
    )

    synchronize_device()
    complete_fold_start = time.perf_counter()

    # -------------------------------------------------------
    # DataLoader creation time
    # -------------------------------------------------------
    loader_start = time.perf_counter()

    train_loader, val_loader, test_loader = (
        create_fold_loaders(
            fold,
            batch_size=CONFIG.batch_size,
        )
    )

    loader_setup_seconds = (
        time.perf_counter() - loader_start
    )

    # -------------------------------------------------------
    # Model and optimizer setup time
    # -------------------------------------------------------
    synchronize_device()
    model_setup_start = time.perf_counter()

    model = build_model(DEVICE)

    criterion = nn.CrossEntropyLoss(
        label_smoothing=CONFIG.label_smoothing
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CONFIG.learning_rate,
        weight_decay=CONFIG.weight_decay,
    )

    synchronize_device()

    model_setup_seconds = (
        time.perf_counter() - model_setup_start
    )

    epochs = (
        CONFIG.epochs
        if max_epochs is None
        else int(max_epochs)
    )

    best_val_accuracy = -np.inf
    best_epoch = -1
    epochs_without_improvement = 0

    checkpoint_path = (
        CHECKPOINT_DIR
        / f"fold_{fold_number:02d}_best.pt"
    )

    if checkpoint_path.exists():
        checkpoint_path.unlink()

    history = []

    total_training_seconds = 0.0
    total_validation_seconds = 0.0
    total_checkpoint_save_seconds = 0.0

    # -------------------------------------------------------
    # Training and validation loop
    # -------------------------------------------------------
    for epoch in range(1, epochs + 1):

        (
            train_metrics,
            _,
            _,
            train_seconds,
        ) = run_epoch(
            model=model,
            loader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
        )

        (
            val_metrics,
            _,
            _,
            val_seconds,
        ) = run_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            optimizer=None,
        )

        total_training_seconds += train_seconds
        total_validation_seconds += val_seconds

        history_row = {
            "fold": fold_number,
            "epoch": epoch,

            "train_time_seconds": train_seconds,
            "train_time_minutes": train_seconds / 60,

            "validation_time_seconds": val_seconds,
            "validation_time_minutes": val_seconds / 60,

            "epoch_total_time_seconds": (
                train_seconds + val_seconds
            ),

            "epoch_total_time_minutes": (
                train_seconds + val_seconds
            ) / 60,

            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },

            **{
                f"val_{key}": value
                for key, value
                in val_metrics.items()
            },
        }

        history.append(history_row)

        print(
            f"Fold {fold_number:02d} | "
            f"Epoch {epoch:02d}/{epochs:02d}\n"
            f"  Train: "
            f"loss={train_metrics['loss']:.4f}, "
            f"acc={train_metrics['accuracy']:.4f}, "
            f"time={format_seconds(train_seconds)}\n"
            f"  Val:   "
            f"loss={val_metrics['loss']:.4f}, "
            f"acc={val_metrics['accuracy']:.4f}, "
            f"macro-F1={val_metrics['f1_macro']:.4f}, "
            f"time={format_seconds(val_seconds)}\n"
            f"  Epoch total: "
            f"{format_seconds(train_seconds + val_seconds)}"
        )

        if (
            val_metrics["accuracy"]
            > best_val_accuracy
        ):
            best_val_accuracy = float(
                val_metrics["accuracy"]
            )

            best_epoch = epoch
            epochs_without_improvement = 0

            checkpoint = {
                "fold": fold_number,
                "epoch": int(epoch),

                "best_val_accuracy": float(
                    best_val_accuracy
                ),

                "model_state_dict": (
                    model.state_dict()
                ),

                "optimizer_state_dict": (
                    optimizer.state_dict()
                ),

                "val_metrics": {
                    key: (
                        float(value)
                        if isinstance(
                            value,
                            (
                                float,
                                int,
                                np.floating,
                                np.integer,
                            ),
                        )
                        else value
                    )
                    for key, value
                    in val_metrics.items()
                },

                "config": asdict(CONFIG),
            }

            synchronize_device()
            checkpoint_save_start = time.perf_counter()

            torch.save(
                checkpoint,
                checkpoint_path,
            )

            synchronize_device()

            checkpoint_save_seconds = (
                time.perf_counter()
                - checkpoint_save_start
            )

            total_checkpoint_save_seconds += (
                checkpoint_save_seconds
            )

        else:
            epochs_without_improvement += 1

        if (
            epochs_without_improvement
            >= CONFIG.early_stopping_patience
        ):
            print(
                f"\nEarly stopping at epoch {epoch}. "
                f"Best epoch: {best_epoch}"
            )
            break

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"No checkpoint was created: "
            f"{checkpoint_path}"
        )

    # -------------------------------------------------------
    # Best checkpoint loading time
    # -------------------------------------------------------
    synchronize_device()
    checkpoint_load_start = time.perf_counter()

    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    synchronize_device()

    checkpoint_load_seconds = (
        time.perf_counter()
        - checkpoint_load_start
    )

    # -------------------------------------------------------
    # Test time
    # -------------------------------------------------------
    (
        test_metrics,
        y_true,
        y_pred,
        test_seconds,
    ) = run_epoch(
        model=model,
        loader=test_loader,
        criterion=criterion,
        optimizer=None,
    )

    synchronize_device()

    complete_fold_seconds = (
        time.perf_counter()
        - complete_fold_start
    )

    history_df = pd.DataFrame(history)

    result = {
        "fold": fold_number,
        "epochs_completed": len(history),
        "best_epoch": int(best_epoch),
        "best_val_accuracy": float(
            best_val_accuracy
        ),

        # Setup timings
        "loader_setup_seconds": (
            loader_setup_seconds
        ),
        "model_setup_seconds": (
            model_setup_seconds
        ),

        # Main timings
        "total_training_seconds": (
            total_training_seconds
        ),
        "total_training_minutes": (
            total_training_seconds / 60
        ),

        "total_validation_seconds": (
            total_validation_seconds
        ),
        "total_validation_minutes": (
            total_validation_seconds / 60
        ),

        "checkpoint_save_seconds": (
            total_checkpoint_save_seconds
        ),

        "checkpoint_load_seconds": (
            checkpoint_load_seconds
        ),

        "test_seconds": test_seconds,
        "test_minutes": test_seconds / 60,

        "complete_fold_seconds": (
            complete_fold_seconds
        ),
        "complete_fold_minutes": (
            complete_fold_seconds / 60
        ),

        **{
            f"test_{key}": value
            for key, value
            in test_metrics.items()
        },
    }

    print("\n" + "=" * 70)
    print(f"Fold {fold_number:02d} timing summary")
    print("=" * 70)

    print(
        "Total training time:   ",
        format_seconds(total_training_seconds),
    )

    print(
        "Total validation time: ",
        format_seconds(total_validation_seconds),
    )

    print(
        "Test time:             ",
        format_seconds(test_seconds),
    )

    print(
        "Checkpoint save time:  ",
        format_seconds(
            total_checkpoint_save_seconds
        ),
    )

    print(
        "Checkpoint load time:  ",
        format_seconds(
            checkpoint_load_seconds
        ),
    )

    print(
        "Complete fold runtime: ",
        format_seconds(
            complete_fold_seconds
        ),
    )

    del model
    del optimizer

    gc.collect()

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return (
        result,
        history_df,
        y_true,
        y_pred,
    )


print(
    "Cell 30 complete: detailed training, validation, "
    "testing, and total runtime measurement is enabled."
)

In [ ]:
# Cell 31: Run a two-epoch timed sanity check

import gc
import pandas as pd
import torch

from sklearn.metrics import confusion_matrix


sanity_checkpoint = (
    CHECKPOINT_DIR
    / "fold_01_best.pt"
)

if sanity_checkpoint.exists():
    sanity_checkpoint.unlink()


sanity_result, sanity_history, sanity_true, sanity_pred = (
    train_one_fold(
        folds[0],
        max_epochs=2,
    )
)


print("\nSanity test result:")
display(
    pd.DataFrame([sanity_result])
)


print("\nPer-epoch timing and performance:")

timing_columns = [
    "epoch",

    "train_loss",
    "train_accuracy",
    "train_time_seconds",
    "train_time_minutes",

    "val_loss",
    "val_accuracy",
    "val_f1_macro",
    "validation_time_seconds",
    "validation_time_minutes",

    "epoch_total_time_seconds",
    "epoch_total_time_minutes",
]

available_columns = [
    column
    for column in timing_columns
    if column in sanity_history.columns
]

display(
    sanity_history[available_columns]
)


print("\nConfusion matrix:")
print(
    confusion_matrix(
        sanity_true,
        sanity_pred,
    )
)


print("\nComplete timing summary:")

print(
    "Total training time:",
    format_seconds(
        sanity_result[
            "total_training_seconds"
        ]
    ),
)

print(
    "Total validation time:",
    format_seconds(
        sanity_result[
            "total_validation_seconds"
        ]
    ),
)

print(
    "Test time:",
    format_seconds(
        sanity_result["test_seconds"]
    ),
)

print(
    "Complete fold time:",
    format_seconds(
        sanity_result[
            "complete_fold_seconds"
        ]
    ),
)


# Estimate the maximum 30-epoch runtime from average observed times
epochs_completed = sanity_result[
    "epochs_completed"
]

average_train_seconds = (
    sanity_result["total_training_seconds"]
    / epochs_completed
)

average_val_seconds = (
    sanity_result["total_validation_seconds"]
    / epochs_completed
)

estimated_30_epoch_fold_seconds = (
    30
    * (
        average_train_seconds
        + average_val_seconds
    )
    + sanity_result["test_seconds"]
)

estimated_10_fold_seconds = (
    estimated_30_epoch_fold_seconds
    * 10
)


print("\nApproximate maximum runtime estimate:")

print(
    "One fold × 30 epochs:",
    format_seconds(
        estimated_30_epoch_fold_seconds
    ),
)

print(
    "Ten folds × 30 epochs:",
    format_seconds(
        estimated_10_fold_seconds
    ),
)

print(
    "\nActual full runtime may be lower "
    "because early stopping can stop before 30 epochs."
)


# Remove sanity checkpoint before full CV
if sanity_checkpoint.exists():
    sanity_checkpoint.unlink()


del sanity_history
del sanity_true
del sanity_pred

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
# Cell 32: Run complete 10-fold cross-subject CV
# Includes resume, partial saving, fold timing, and ETA

import gc
import time
import pandas as pd
import torch


RUN_FULL_CV = True


if not RUN_FULL_CV:
    print("Full CV is disabled.")
    print("Set RUN_FULL_CV = True and rerun this cell.")

else:
    partial_results_path = (
        RESULT_DIR / "fold_results_partial.csv"
    )

    partial_history_path = (
        RESULT_DIR / "training_history_partial.csv"
    )

    partial_predictions_path = (
        RESULT_DIR / "test_predictions_partial.csv"
    )

    final_results_path = (
        RESULT_DIR / "fold_results.csv"
    )

    final_history_path = (
        RESULT_DIR / "training_history.csv"
    )

    final_predictions_path = (
        RESULT_DIR / "test_predictions.csv"
    )

    # -------------------------------------------------------
    # Load existing partial results for resume
    # -------------------------------------------------------
    if partial_results_path.exists():
        existing_results_df = pd.read_csv(
            partial_results_path
        )

        completed_folds = set(
            existing_results_df["fold"]
            .astype(int)
            .tolist()
        )

        fold_results = (
            existing_results_df
            .to_dict("records")
        )

        print("Resume mode enabled.")
        print(
            "Completed folds:",
            sorted(completed_folds),
        )

    else:
        completed_folds = set()
        fold_results = []

        print("Starting a new full CV run.")

    if partial_history_path.exists():
        all_histories = [
            pd.read_csv(
                partial_history_path
            )
        ]
    else:
        all_histories = []

    if partial_predictions_path.exists():
        all_predictions = [
            pd.read_csv(
                partial_predictions_path
            )
        ]
    else:
        all_predictions = []

    total_fold_count = len(folds)
    remaining_folds = [
        fold
        for fold in folds
        if int(fold["fold"])
        not in completed_folds
    ]

    print(
        f"Total folds: {total_fold_count}"
    )

    print(
        f"Remaining folds: {len(remaining_folds)}"
    )

    print(
        f"Maximum epochs per fold: "
        f"{CONFIG.epochs}"
    )

    print(
        f"Early stopping patience: "
        f"{CONFIG.early_stopping_patience}"
    )

    # Sanity-run-based initial estimate
    estimated_seconds_per_fold = (
        4.08 * 3600
    )

    initial_estimated_seconds = (
        estimated_seconds_per_fold
        * len(remaining_folds)
    )

    print(
        "\nInitial worst-case estimate:",
        format_seconds(
            initial_estimated_seconds
        ),
    )

    print(
        "Actual runtime should be lower if "
        "early stopping activates."
    )

    synchronize_device()
    cv_session_start = time.perf_counter()

    completed_this_session = 0
    observed_fold_times = []

    # -------------------------------------------------------
    # Run each fold
    # -------------------------------------------------------
    for fold in folds:
        fold_number = int(
            fold["fold"]
        )

        if fold_number in completed_folds:
            print(
                f"\nSkipping fold {fold_number}: "
                "already completed."
            )
            continue

        print("\n" + "=" * 100)

        print(
            f"Starting fold {fold_number}/"
            f"{total_fold_count}"
        )

        print("=" * 100)

        synchronize_device()
        fold_start = time.perf_counter()

        try:
            (
                result,
                history_df,
                y_true,
                y_pred,
            ) = train_one_fold(fold)

            synchronize_device()

            fold_wall_seconds = (
                time.perf_counter()
                - fold_start
            )

            result[
                "fold_wall_seconds"
            ] = fold_wall_seconds

            result[
                "fold_wall_minutes"
            ] = fold_wall_seconds / 60

            result[
                "fold_wall_hours"
            ] = fold_wall_seconds / 3600

            fold_results.append(result)
            all_histories.append(
                history_df
            )

            prediction_df = pd.DataFrame(
                {
                    "fold": fold_number,
                    "y_true": y_true,
                    "y_pred": y_pred,
                }
            )

            all_predictions.append(
                prediction_df
            )

            completed_folds.add(
                fold_number
            )

            completed_this_session += 1
            observed_fold_times.append(
                fold_wall_seconds
            )

            # ------------------------------------------------
            # Save all partial outputs after each fold
            # ------------------------------------------------
            current_results_df = (
                pd.DataFrame(
                    fold_results
                )
                .sort_values("fold")
                .drop_duplicates(
                    subset=["fold"],
                    keep="last",
                )
            )

            current_history_df = (
                pd.concat(
                    all_histories,
                    ignore_index=True,
                )
                .drop_duplicates(
                    subset=[
                        "fold",
                        "epoch",
                    ],
                    keep="last",
                )
                .sort_values(
                    [
                        "fold",
                        "epoch",
                    ]
                )
            )

            current_predictions_df = (
                pd.concat(
                    all_predictions,
                    ignore_index=True,
                )
            )

            current_results_df.to_csv(
                partial_results_path,
                index=False,
            )

            current_history_df.to_csv(
                partial_history_path,
                index=False,
            )

            current_predictions_df.to_csv(
                partial_predictions_path,
                index=False,
            )

            print(
                f"\nFold {fold_number} completed."
            )

            print(
                "Fold runtime:",
                format_seconds(
                    fold_wall_seconds
                ),
            )

            print(
                "Epochs completed:",
                result[
                    "epochs_completed"
                ],
            )

            print(
                "Best epoch:",
                result[
                    "best_epoch"
                ],
            )

            print(
                "Test accuracy:",
                f"{100 * result['test_accuracy']:.2f}%",
            )

            print(
                "Test macro-F1:",
                f"{100 * result['test_f1_macro']:.2f}%",
            )

            # -----------------------------------------------
            # Dynamic ETA based on completed folds
            # -----------------------------------------------
            remaining_count = (
                total_fold_count
                - len(completed_folds)
            )

            average_observed_seconds = (
                sum(observed_fold_times)
                / len(observed_fold_times)
            )

            estimated_remaining_seconds = (
                average_observed_seconds
                * remaining_count
            )

            print(
                "Completed folds:",
                len(completed_folds),
                "/",
                total_fold_count,
            )

            print(
                "Estimated remaining time:",
                format_seconds(
                    estimated_remaining_seconds
                ),
            )

        except Exception as error:
            print(
                f"\nFold {fold_number} failed."
            )

            print(
                "Error type:",
                type(error).__name__,
            )

            print(
                "Error message:",
                str(error),
            )

            print(
                "\nAll previously completed folds "
                "are saved in partial CSV files."
            )

            raise

        finally:
            gc.collect()

            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    # -------------------------------------------------------
    # End session timer
    # -------------------------------------------------------
    synchronize_device()

    cv_session_seconds = (
        time.perf_counter()
        - cv_session_start
    )

    # -------------------------------------------------------
    # Build final outputs
    # -------------------------------------------------------
    fold_results_df = (
        pd.DataFrame(
            fold_results
        )
        .sort_values("fold")
        .drop_duplicates(
            subset=["fold"],
            keep="last",
        )
        .reset_index(drop=True)
    )

    histories_df = (
        pd.concat(
            all_histories,
            ignore_index=True,
        )
        .drop_duplicates(
            subset=[
                "fold",
                "epoch",
            ],
            keep="last",
        )
        .sort_values(
            [
                "fold",
                "epoch",
            ]
        )
        .reset_index(drop=True)
    )

    predictions_df = (
        pd.concat(
            all_predictions,
            ignore_index=True,
        )
        .reset_index(drop=True)
    )

    fold_results_df.to_csv(
        final_results_path,
        index=False,
    )

    histories_df.to_csv(
        final_history_path,
        index=False,
    )

    predictions_df.to_csv(
        final_predictions_path,
        index=False,
    )

    # -------------------------------------------------------
    # Save session and cumulative timing
    # -------------------------------------------------------
    cumulative_fold_seconds = (
        fold_results_df[
            "complete_fold_seconds"
        ].sum()
        if "complete_fold_seconds"
        in fold_results_df.columns
        else np.nan
    )

    runtime_df = pd.DataFrame(
        [
            {
                "completed_folds": (
                    len(fold_results_df)
                ),
                "expected_folds": (
                    total_fold_count
                ),
                "folds_completed_this_session": (
                    completed_this_session
                ),
                "this_session_seconds": (
                    cv_session_seconds
                ),
                "this_session_minutes": (
                    cv_session_seconds / 60
                ),
                "this_session_hours": (
                    cv_session_seconds / 3600
                ),
                "cumulative_fold_seconds": (
                    cumulative_fold_seconds
                ),
                "cumulative_fold_hours": (
                    cumulative_fold_seconds / 3600
                    if pd.notna(
                        cumulative_fold_seconds
                    )
                    else np.nan
                ),
            }
        ]
    )

    runtime_df.to_csv(
        RESULT_DIR / "full_cv_runtime.csv",
        index=False,
    )

    print("\n" + "=" * 100)
    print("Cross-validation execution finished")
    print("=" * 100)

    print(
        "Completed folds:",
        len(fold_results_df),
        "/",
        total_fold_count,
    )

    print(
        "Current session runtime:",
        format_seconds(
            cv_session_seconds
        ),
    )

    if pd.notna(
        cumulative_fold_seconds
    ):
        print(
            "Cumulative fold runtime:",
            format_seconds(
                cumulative_fold_seconds
            ),
        )

    print("\nSaved files:")

    print(
        final_results_path
    )

    print(
        final_history_path
    )

    print(
        final_predictions_path
    )

    print(
        RESULT_DIR
        / "full_cv_runtime.csv"
    )

    display(
        fold_results_df
    )

In [ ]:
# Cell 33: Summarize full cross-validation metrics and runtime

from pathlib import Path
import numpy as np
import pandas as pd


fold_results_path = RESULT_DIR / "fold_results.csv"

if not fold_results_path.exists():
    print(
        "Full fold-results file does not exist yet. "
        "Run Cell 32 first."
    )

else:
    fold_results_df = pd.read_csv(
        fold_results_path
    )

    print(
        f"Loaded {len(fold_results_df)} fold result(s)."
    )

    if len(fold_results_df) < CONFIG.num_test_folds:
        print(
            f"Warning: expected {CONFIG.num_test_folds} folds, "
            f"but found only {len(fold_results_df)}."
        )
        print(
            "This summary is incomplete and should not be "
            "reported as the final 10-fold result."
        )

    # -------------------------------------------------------
    # Classification metrics
    # -------------------------------------------------------
    requested_metric_columns = [
        "test_accuracy",
        "test_f1_binary",
        "test_f1_macro",
        "test_f1_weighted",
        "test_precision_binary",
        "test_recall_binary",
    ]

    metric_columns = [
        column
        for column in requested_metric_columns
        if column in fold_results_df.columns
    ]

    missing_metric_columns = [
        column
        for column in requested_metric_columns
        if column not in fold_results_df.columns
    ]

    if missing_metric_columns:
        print(
            "\nMissing metric columns:",
            missing_metric_columns,
        )

    metric_summary_rows = []

    for column in metric_columns:
        values = pd.to_numeric(
            fold_results_df[column],
            errors="coerce",
        ).dropna()

        if len(values) == 0:
            continue

        mean_value = values.mean()

        std_value = (
            values.std(ddof=1)
            if len(values) > 1
            else np.nan
        )

        metric_summary_rows.append(
            {
                "metric": column,
                "folds_available": len(values),
                "mean": mean_value,
                "std": std_value,
                "mean_percent": 100 * mean_value,
                "std_percent": (
                    100 * std_value
                    if pd.notna(std_value)
                    else np.nan
                ),
                "min_percent": 100 * values.min(),
                "max_percent": 100 * values.max(),
            }
        )

    cv_summary_df = pd.DataFrame(
        metric_summary_rows
    )

    cv_summary_df.to_csv(
        RESULT_DIR / "cv_summary.csv",
        index=False,
    )

    # -------------------------------------------------------
    # Runtime summary
    # -------------------------------------------------------
    requested_runtime_columns = [
        "epochs_completed",
        "total_training_seconds",
        "total_validation_seconds",
        "test_seconds",
        "checkpoint_save_seconds",
        "checkpoint_load_seconds",
        "loader_setup_seconds",
        "model_setup_seconds",
        "complete_fold_seconds",
    ]

    runtime_columns = [
        column
        for column in requested_runtime_columns
        if column in fold_results_df.columns
    ]

    runtime_summary_rows = []

    for column in runtime_columns:
        values = pd.to_numeric(
            fold_results_df[column],
            errors="coerce",
        ).dropna()

        if len(values) == 0:
            continue

        if column == "epochs_completed":
            runtime_summary_rows.append(
                {
                    "measure": column,
                    "folds_available": len(values),
                    "mean": values.mean(),
                    "std": (
                        values.std(ddof=1)
                        if len(values) > 1
                        else np.nan
                    ),
                    "total": values.sum(),
                    "unit": "epochs",
                }
            )

        else:
            runtime_summary_rows.append(
                {
                    "measure": column,
                    "folds_available": len(values),
                    "mean": values.mean(),
                    "std": (
                        values.std(ddof=1)
                        if len(values) > 1
                        else np.nan
                    ),
                    "total": values.sum(),
                    "unit": "seconds",
                }
            )

    runtime_summary_df = pd.DataFrame(
        runtime_summary_rows
    )

    runtime_summary_df.to_csv(
        RESULT_DIR / "runtime_summary.csv",
        index=False,
    )

    # -------------------------------------------------------
    # Human-readable runtime totals
    # -------------------------------------------------------
    total_training_seconds = (
        fold_results_df[
            "total_training_seconds"
        ].sum()
        if "total_training_seconds"
        in fold_results_df.columns
        else np.nan
    )

    total_validation_seconds = (
        fold_results_df[
            "total_validation_seconds"
        ].sum()
        if "total_validation_seconds"
        in fold_results_df.columns
        else np.nan
    )

    total_test_seconds = (
        fold_results_df[
            "test_seconds"
        ].sum()
        if "test_seconds"
        in fold_results_df.columns
        else np.nan
    )

    total_complete_seconds = (
        fold_results_df[
            "complete_fold_seconds"
        ].sum()
        if "complete_fold_seconds"
        in fold_results_df.columns
        else np.nan
    )

    # -------------------------------------------------------
    # Display
    # -------------------------------------------------------
    print("\nPer-fold results:")
    display(fold_results_df)

    print("\nCross-validation metric summary:")
    display(cv_summary_df)

    print("\nRuntime summary:")
    display(runtime_summary_df)

    print("\nOverall runtime totals:")

    if pd.notna(total_training_seconds):
        print(
            "Total training time:",
            format_seconds(
                total_training_seconds
            ),
        )

    if pd.notna(total_validation_seconds):
        print(
            "Total validation time:",
            format_seconds(
                total_validation_seconds
            ),
        )

    if pd.notna(total_test_seconds):
        print(
            "Total test time:",
            format_seconds(
                total_test_seconds
            ),
        )

    if pd.notna(total_complete_seconds):
        print(
            "Complete 10-fold runtime:",
            format_seconds(
                total_complete_seconds
            ),
        )

    # -------------------------------------------------------
    # Reference comparison
    # -------------------------------------------------------
    print("\nPublished MSGM FACED reference target:")
    print("Accuracy: 63.17 ± 3.62%")
    print("Reported F1: 76.01 ± 3.74%")

    print(
        "\nImportant: compare the reported F1 against "
        "binary, macro, and weighted F1 separately because "
        "the exact averaging definition must be verified "
        "before claiming a direct reproduction."
    )

    print("\nSaved files:")
    print(
        RESULT_DIR / "cv_summary.csv"
    )
    print(
        RESULT_DIR / "runtime_summary.csv"
    )

In [ ]:
# Cell 34: Generate the pooled confusion matrix and classification report

predictions_path = RESULT_DIR / "test_predictions.csv"

if not predictions_path.exists():
    print("Prediction file does not exist yet. Run Cell 32 first.")
else:
    predictions_df = pd.read_csv(predictions_path)
    y_true = predictions_df["y_true"].to_numpy()
    y_pred = predictions_df["y_pred"].to_numpy()

    print("Pooled confusion matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nPooled classification report:")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["negative", "positive"],
            digits=4,
            zero_division=0,
        )
    )

In [ ]:
# Cell 35: Package all reproducibility outputs for download

import shutil

archive_base = Path("/kaggle/working/faced_msgm_reproduction_outputs")
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=WORK_DIR,
)

print("Created archive:")
print(archive_path)
print("\nDownload this ZIP from the Kaggle Output panel.")